In [ ]:
testing_indices_2 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_2_soliton_b_low = []

print("Starting 2-Soliton Experiment momentum and energy (low weight) Experiment...")

for testing_val in testing_indices_2:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons=2,
            n_hidden_layers=7, 
            n_neurons_per_layer=62, 
            activation=nn.Tanh,
            seed=current_seed, 
            verbose=False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 100000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 2.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 10.0,    
            w_bc                     = 1.0,    
            w_pde                    = 100.0,
            w_momentum               = 0.05,
            w_energy                 = 0.05,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_2_soliton_b_low.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('2_soliton_b_low_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_2_soliton_b_low, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_2_soliton_b_low[-1]['mae_mean']:.6e} ===\n")
        
print("2 Soliton Experiment LOW MOMENTUM AND ENERGY OVER!")